# Atomsk Structure Building

Atomsk is a command-line tool for creating, transforming, and converting atomistic structures. This notebook introduces common Atomsk commands for building structures that can be imported into LAMMPS and visualized in OVITO.

## 1. Why Use Atomsk?

LAMMPS can create many simple structures internally, but Atomsk is especially useful for:

- Converting between structure file formats.
- Creating oriented crystals.
- Building defects, surfaces, and interfaces.
- Preparing LAMMPS data files from external structures.

In [ ]:
from pathlib import Path

atomsk_dir = Path("atomsk")
atomsk_dir.mkdir(exist_ok=True)

commands = """# Example Atomsk commands for this lesson

# Build an FCC aluminum unit cell and write a LAMMPS data file
atomsk --create fcc 4.05 Al Al_fcc.xsf
atomsk Al_fcc.xsf -duplicate 8 8 8 Al_fcc_8x8x8.lmp

# Build an oriented BCC iron structure
atomsk --create bcc 2.866 Fe orient [100] [010] [001] Fe_bcc.xsf
atomsk Fe_bcc.xsf -duplicate 10 10 10 Fe_bcc_10x10x10.lmp

# Create a vacancy by removing one atom near the cell center
atomsk Al_fcc_8x8x8.lmp -select in sphere 16.2 16.2 16.2 1.0 -remove-atoms select Al_vacancy.lmp

# Convert a LAMMPS data file to XYZ for quick viewing
atomsk Al_fcc_8x8x8.lmp Al_fcc_8x8x8.xyz
"""

command_path = atomsk_dir / "example_atomsk_commands.txt"
command_path.write_text(commands)
print(command_path.resolve())

In [ ]:
print(commands)

## 2. Connecting Atomsk to LAMMPS

After Atomsk creates a LAMMPS data file, LAMMPS can read it with `read_data`:

```lammps
units metal
atom_style atomic
boundary p p p
read_data atomsk/Al_fcc_8x8x8.lmp
```

The interatomic potential still needs to be supplied separately in the LAMMPS input script.

In [ ]:
lammps_import = """# LAMMPS skeleton for an Atomsk-generated Al structure
units           metal
dimension       3
boundary        p p p
atom_style      atomic

read_data       atomsk/Al_fcc_8x8x8.lmp

# Replace this with a real Al EAM potential file available on your system.
pair_style      eam/alloy
pair_coeff      * * Al99.eam.alloy Al

thermo          100
minimize        1.0e-8 1.0e-10 1000 10000
"""

lammps_dir = Path("lammps")
lammps_dir.mkdir(exist_ok=True)
(lammps_dir / "in.import_atomsk_al").write_text(lammps_import)
print(lammps_import)

## Exercises

1. Create FCC Cu, BCC Fe, and diamond Si structures.
2. Use `-duplicate` to compare small and large simulation cells.
3. Convert a structure to XYZ and open it in OVITO.
4. Create one vacancy and compare the atom count before and after.
5. Prepare a LAMMPS data file and run an energy minimization.